# Gold: lab_test_trends

Builds `<catalog>.gold.lab_test_trends` per the Silver-to-Gold STTM: one row per test type per month with abnormal/critical rates and a month-over-month trend (LAG). Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);

CREATE OR REPLACE TABLE gold.lab_test_trends
COMMENT 'One row per test type per month: volume, abnormal/critical rates and month-over-month trend'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                            AS year,
    month(test_date)                                           AS month,
    COUNT(*)                                                   AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)     AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)     AS critical_test_count
  FROM silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  r.test_name,
  CAST(r.year AS INT)                                        AS year,
  CAST(r.month AS INT)                                       AS month,
  c.month_name,
  c.quarter_name,
  CAST(r.total_tests AS INT)                                 AS total_tests,
  CAST(r.abnormal_test_count AS INT)                         AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)                         AS critical_test_count,
  r.abnormal_rate_pct,
  r.critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))       AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN silver.calendar c
  ON c.date_key = CAST(date_format(make_date(r.year, r.month, 1), 'yyyyMMdd') AS INT);

In [0]:
%sql
SELECT
  COUNT(*)                                   AS gold_rows,
  COUNT(DISTINCT test_name, year, month)     AS distinct_test_months,
  SUM(total_tests)                           AS total_tests,
  (SELECT COUNT(*) FROM silver.lab_results)  AS source_lab_results,
  SUM(CASE WHEN month_name IS NULL THEN 1 ELSE 0 END) AS missing_calendar
FROM gold.lab_test_trends